Save base Logit, KNN, SVC, LogisticRegression, CatBoost, XGboost with pipeline. Sedimentation_Stk feature was used!

Based on `research_14_SMOTE_influence_on_metrics.ipynb` it was decided to use SMOTE for all models, except Logit!

base_estimator_params - redundant thing in logging! - SOLVED!

# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

import joblib

from statsmodels.api import Logit
from statsmodels.api import Logit # type: ignore
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

from sklearn.metrics import (
    make_scorer,
    f1_score,
)

import optuna
from functools import partial

from pathlib import Path
import sys

from typing import Type
from sklearn.base import BaseEstimator

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling4_utils import (
    # _prepare_df,
    MLPipeline,
    StatsModelsEstimator,
    update_estimator_params,
    OptunaOptimizer,
    GridSearchOptimizer,
    RANDOM_STATE,
)

seed = RANDOM_STATE

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Settings

In [4]:
model_postfix = 'opt-model'
add_smote = True
is_smotenc = False
smote_params = {
    # 'categorical_features': (
    #     'wettability',
    #     'inclination',
    # ),
}
scoring_metrics = {"f1_macro": make_scorer(f1_score, average="macro"),}
step_scoring_average = "mean"
n_trials = 1500 # checked with 15 and 100

save_model_and_metrics = True
metrics_file = "metrics_modelling4_2.2-pure_smote_with_ensembles.xlsx"

## Optimization functions

In [5]:
def optimize_estimator_optuna(
    target:str,
    estimator:Type[BaseEstimator],
    estimator_params:dict,
    objective:callable,
    n_trials:int,
    base_estimator:Type[BaseEstimator]=None,
    base_estimator_params:dict=None,
    step_scoring_average:str=step_scoring_average,
    params_to_process:list=None,
    base_estimator_params_list:list=None,
    model_postfix:str=model_postfix,
    add_smote:bool=add_smote,
    is_smotenc:bool=is_smotenc,
    smote_params:dict=smote_params,
    scoring_metrics:dict=scoring_metrics,
    save_model_and_metrics:bool=save_model_and_metrics,
    metrics_file:str=metrics_file,
    seed:int=seed,
):
    
    estimator_params = estimator_params.copy()
    
    # Switch off probability for SVC, since it is long to optimize
    if 'probability' in estimator_params:
        estimator_params['probability'] = False
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        base_estimator=base_estimator,
        base_estimator_params=base_estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        scoring_metrics=scoring_metrics,
        step_scoring_average=step_scoring_average,
    )
    
    opt = OptunaOptimizer(
        objective=partial(objective, ml_pipe=ml_pipe),
        study_name="model_study",
        direction="maximize",
        seed=seed,
    )
    
    opt.optimize(n_trials=n_trials)
    
    best_params = opt.study.best_params
    
    if params_to_process:
        for param in params_to_process:
            # Find one key in best_params which contains param
            key = next((k for k in best_params.keys() if param in k), None)
            if key:
                best_params[param] = best_params.pop(key)

    # Process base_estimator_params
    if base_estimator_params_list:
        best_params_base_estimator = {}
        for param in base_estimator_params_list:
            # Find one key in best_params which contains param
            key = next((k for k in best_params.keys() if param in k), None)
            if key:
                best_params_base_estimator[param] = best_params.pop(key)
        print('best_params for main estimator')
        display(best_params)
        print('best_params for base estimator')
        display(best_params_base_estimator)
        if base_estimator_params is None:
            base_estimator_params = best_params_base_estimator
        else:
            base_estimator_params = base_estimator_params.copy()
            base_estimator_params.update(best_params_base_estimator)
    else:
        print('best_params')
        display(best_params)
    
    # Switch on probability for SVC, to get metrics like roc_auc for the final model
    if 'probability' in estimator_params:
        estimator_params['probability'] = True
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params={**estimator_params, **best_params},
        base_estimator=base_estimator,
        base_estimator_params=base_estimator_params, # Would be None if base_estimator_params_list is not provided
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        step_scoring_average=step_scoring_average, # No need to pass, it would not be used
        metrics_file=metrics_file,
    )
    
    ml_pipe.run(
        save_model_and_metrics=save_model_and_metrics,
    )

# CatBoost

In [6]:
estimator = CatBoostClassifier
target = 'splashing'
# TODO: Try with GPU!
estimator_params = {
    'verbose': False,
}
# params_to_process = ['gamma']
params_to_process = None

def catboost_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):

    suggested_estimator_params = {
        "iterations": trial.suggest_int("iterations", 50, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "depth": trial.suggest_int("depth", 3, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.0, 1.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "grow_policy": trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]),
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]),
        "auto_class_weights": trial.suggest_categorical("auto_class_weights", ["Balanced", "SqrtBalanced", None]),
    }
    
    if suggested_estimator_params['bootstrap_type'] == 'Bernoulli':
        suggested_estimator_params['subsample'] = trial.suggest_float(
            "subsample", 0.5, 1.0
        )
        
    if suggested_estimator_params['bootstrap_type'] == 'Bayesian':
        suggested_estimator_params['bagging_temperature'] = (
            trial.suggest_float("bagging_temperature", 0.0, 1.0)
        )

    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params,
    )
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=catboost_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-15 20:21:30,698] A new study created in memory with name: model_study
[I 2025-04-15 20:21:37,714] Trial 0 finished with value: 0.8788453856583421 and parameters: {'iterations': 406, 'learning_rate': 0.22648248189516848, 'depth': 8, 'l2_leaf_reg': 0.24810409748678125, 'random_strength': 0.15601864044243652, 'border_count': 66, 'grow_policy': 'Depthwise', 'bootstrap_type': 'MVS', 'auto_class_weights': 'Balanced'}. Best is trial 0 with value: 0.8788453856583421.
[I 2025-04-15 20:21:41,229] Trial 1 finished with value: 0.8889425016414505 and parameters: {'iterations': 224, 'learning_rate': 0.005670807781371429, 'depth': 7, 'l2_leaf_reg': 0.05342937261279776, 'random_strength': 0.2912291401980419, 'border_count': 169, 'grow_policy': 'Lossguide', 'bootstrap_type': 'Bernoulli', 'auto_class_weights': 'SqrtBalanced', 'subsample': 0.8037724259507192}. Best is trial 1 with value: 0.8889425016414505.
[I 2025-04-15 20:21:51,839] Trial 2 finished with value: 0.8725962524773413 and paramet

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "CatBoostClassifier"


,0
target,splashing
model,CatBoostClassifier_splashing_smote_opt-model
holdout_test_f1_macro,0.882261
holdout_test_accuracy_balanced,0.876157
holdout_test_roc_auc,0.957562
holdout_test_f1,0.918367
holdout_test_accuracy,0.893333
cv_test_f1_macro_median,0.898584
cv_test_accuracy_balanced_median,0.903251
cv_test_roc_auc_median,0.958204


Model saved in ../results/models_modelling4/CatBoostClassifier_splashing_smote_opt-model


In [7]:
estimator = CatBoostClassifier
target = 'no_fragmentation'
estimator_params = {
    'verbose': False,
}
# params_to_process = ['gamma']
params_to_process = None

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=catboost_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-15 20:22:47,788] A new study created in memory with name: model_study
[I 2025-04-15 20:22:57,794] Trial 0 finished with value: 0.8419324527019028 and parameters: {'iterations': 406, 'learning_rate': 0.22648248189516848, 'depth': 8, 'l2_leaf_reg': 0.24810409748678125, 'random_strength': 0.15601864044243652, 'border_count': 66, 'grow_policy': 'Depthwise', 'bootstrap_type': 'MVS', 'auto_class_weights': 'Balanced'}. Best is trial 0 with value: 0.8419324527019028.
[I 2025-04-15 20:23:01,214] Trial 1 finished with value: 0.8567647997561406 and parameters: {'iterations': 224, 'learning_rate': 0.005670807781371429, 'depth': 7, 'l2_leaf_reg': 0.05342937261279776, 'random_strength': 0.2912291401980419, 'border_count': 169, 'grow_policy': 'Lossguide', 'bootstrap_type': 'Bernoulli', 'auto_class_weights': 'SqrtBalanced', 'subsample': 0.8037724259507192}. Best is trial 1 with value: 0.8567647997561406.
[I 2025-04-15 20:23:11,067] Trial 2 finished with value: 0.8383569024074005 and paramet

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "CatBoostClassifier"


,0
target,no_fragmentation
model,CatBoostClassifier_no_fragmentation_smote_opt-...
holdout_test_f1_macro,0.913375
holdout_test_accuracy_balanced,0.906818
holdout_test_roc_auc,0.984545
holdout_test_f1,0.871795
holdout_test_accuracy,0.933333
cv_test_f1_macro_median,0.90293
cv_test_accuracy_balanced_median,0.900183
cv_test_roc_auc_median,0.981197


Model saved in ../results/models_modelling4/CatBoostClassifier_no_fragmentation_smote_opt-model


# XGBoost

In [8]:
estimator = XGBClassifier
target = 'splashing'
estimator_params = {
    'n_jobs': 1,
}
# params_to_process = ['gamma']
params_to_process = None

def xgboost_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):

    # https://xgboost.readthedocs.io/en/latest/parameter.html
    # sum(negative instances) / sum(positive instances)
    scale_pos_weight = 132/240 # For splashing

    suggested_estimator_params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True), # Catboost l2_leaf_reg
        "scale_pos_weight": trial.suggest_categorical("scale_pos_weight", [scale_pos_weight, 1.0]),
    }

    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params,
    )
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=xgboost_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-15 20:24:03,764] A new study created in memory with name: model_study
[I 2025-04-15 20:24:03,969] Trial 0 finished with value: 0.751780256501701 and parameters: {'n_estimators': 406, 'learning_rate': 0.22648248189516848, 'max_depth': 8, 'min_child_weight': 12, 'gamma': 0.7800932022121826, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.5290418060840998, 'reg_alpha': 2.9154431891537547, 'reg_lambda': 0.2537815508265665, 'scale_pos_weight': 0.55}. Best is trial 0 with value: 0.751780256501701.
[I 2025-04-15 20:24:04,405] Trial 1 finished with value: 0.8268852139678102 and parameters: {'n_estimators': 972, 'learning_rate': 0.11536162338241392, 'max_depth': 3, 'min_child_weight': 4, 'gamma': 0.9170225492671691, 'subsample': 0.6521211214797689, 'colsample_bytree': 0.762378215816119, 'reg_alpha': 0.05342937261279776, 'reg_lambda': 0.014618962793704957, 'scale_pos_weight': 0.55}. Best is trial 1 with value: 0.8268852139678102.
[I 2025-04-15 20:24:04,608] Trial 2 finished wit

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "XGBClassifier"


,0
target,splashing
model,XGBClassifier_splashing_smote_opt-model
holdout_test_f1_macro,0.810348
holdout_test_accuracy_balanced,0.80787
holdout_test_roc_auc,0.922068
holdout_test_f1,0.865979
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.817451
cv_test_accuracy_balanced_median,0.841331
cv_test_roc_auc_median,0.941176


Model saved in ../results/models_modelling4/XGBClassifier_splashing_smote_opt-model


In [9]:
estimator = XGBClassifier
target = 'no_fragmentation'
estimator_params = {
    'n_jobs': 1,
}
params_to_process = None

def xgboost_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):

    # https://xgboost.readthedocs.io/en/latest/parameter.html
    # sum(negative instances) / sum(positive instances)
    scale_pos_weight = 273/99 # For no_fragmentation

    suggested_estimator_params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True), # Catboost l2_leaf_reg
        "scale_pos_weight": trial.suggest_categorical("scale_pos_weight", [1.0, scale_pos_weight]),
    }

    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params,
    )
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=xgboost_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-15 20:24:09,252] A new study created in memory with name: model_study
[I 2025-04-15 20:24:09,467] Trial 0 finished with value: 0.8103606040406041 and parameters: {'n_estimators': 406, 'learning_rate': 0.22648248189516848, 'max_depth': 8, 'min_child_weight': 12, 'gamma': 0.7800932022121826, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.5290418060840998, 'reg_alpha': 2.9154431891537547, 'reg_lambda': 0.2537815508265665, 'scale_pos_weight': 1.0}. Best is trial 0 with value: 0.8103606040406041.
[I 2025-04-15 20:24:09,888] Trial 1 finished with value: 0.8448348837400322 and parameters: {'n_estimators': 972, 'learning_rate': 0.11536162338241392, 'max_depth': 3, 'min_child_weight': 4, 'gamma': 0.9170225492671691, 'subsample': 0.6521211214797689, 'colsample_bytree': 0.762378215816119, 'reg_alpha': 0.05342937261279776, 'reg_lambda': 0.014618962793704957, 'scale_pos_weight': 1.0}. Best is trial 1 with value: 0.8448348837400322.
[I 2025-04-15 20:24:10,134] Trial 2 finished wit

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "XGBClassifier"


,0
target,no_fragmentation
model,XGBClassifier_no_fragmentation_smote_opt-model
holdout_test_f1_macro,0.867725
holdout_test_accuracy_balanced,0.879545
holdout_test_roc_auc,0.97
holdout_test_f1,0.809524
holdout_test_accuracy,0.893333
cv_test_f1_macro_median,0.861538
cv_test_accuracy_balanced_median,0.89011
cv_test_roc_auc_median,0.967521


Model saved in ../results/models_modelling4/XGBClassifier_no_fragmentation_smote_opt-model


# AdaBoost

In [10]:
estimator = AdaBoostClassifier
target = 'splashing'
estimator_params = {}
base_estimator = DecisionTreeClassifier
base_estimator_params = {}
# params_to_process = ['gamma']
params_to_process = None
base_estimator_params_list = [
    'max_depth',
    'min_samples_split',
    'min_samples_leaf',
    'class_weight'
]

def adaboost_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):

    suggested_base_estimator_params = {
        "max_depth": trial.suggest_int("max_depth", 1, 10),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20), # Closed to min_child_weight
        "class_weight": trial.suggest_categorical('class_weight', [None, 'balanced'])
    }
    base_estimator_params = update_estimator_params(
        ml_pipe,
        suggested_base_estimator_params,
        estimator_type='base',
    )

    suggested_estimator_params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
    }
    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params,
    )
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        base_estimator=ml_pipe._pipeline_params['base_estimator'],
        estimator_params=estimator_params,
        base_estimator_params=base_estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    base_estimator=base_estimator,
    base_estimator_params=base_estimator_params,
    params_to_process=params_to_process,
    base_estimator_params_list=base_estimator_params_list,
    objective=adaboost_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-15 20:24:14,963] A new study created in memory with name: model_study
[I 2025-04-15 20:24:16,344] Trial 0 finished with value: 0.8644623227823021 and parameters: {'max_depth': 4, 'min_samples_split': 20, 'min_samples_leaf': 15, 'class_weight': None, 'n_estimators': 198, 'learning_rate': 0.0013927723945289009}. Best is trial 0 with value: 0.8644623227823021.
[I 2025-04-15 20:24:22,959] Trial 1 finished with value: 0.8765088953337681 and parameters: {'max_depth': 9, 'min_samples_split': 13, 'min_samples_leaf': 15, 'class_weight': 'balanced', 'n_estimators': 841, 'learning_rate': 0.0033572967053517922}. Best is trial 1 with value: 0.8765088953337681.
[I 2025-04-15 20:24:24,682] Trial 2 finished with value: 0.871533285109738 and parameters: {'max_depth': 2, 'min_samples_split': 5, 'min_samples_leaf': 7, 'class_weight': None, 'n_estimators': 326, 'learning_rate': 0.032781876533976156}. Best is trial 1 with value: 0.8765088953337681.
[I 2025-04-15 20:24:26,218] Trial 3 finished wi

best_params for main estimator


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "AdaBoostClassifier"


,0
target,splashing
model,AdaBoostClassifier_splashing_smote_opt-model
holdout_test_f1_macro,0.802991
holdout_test_accuracy_balanced,0.791667
holdout_test_roc_auc,0.881173
holdout_test_f1,0.871287
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.898584
cv_test_accuracy_balanced_median,0.903251
cv_test_roc_auc_median,0.948916


Model saved in ../results/models_modelling4/AdaBoostClassifier_splashing_smote_opt-model


In [11]:
estimator = AdaBoostClassifier
target = 'no_fragmentation'
estimator_params = {}
base_estimator = DecisionTreeClassifier
base_estimator_params = {}
# params_to_process = ['gamma']
params_to_process = None
base_estimator_params_list = [
    'max_depth',
    'min_samples_split',
    'min_samples_leaf',
    'class_weight'
]

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    base_estimator=base_estimator,
    base_estimator_params=base_estimator_params,
    params_to_process=params_to_process,
    base_estimator_params_list=base_estimator_params_list,
    objective=adaboost_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-15 20:25:22,994] A new study created in memory with name: model_study
[I 2025-04-15 20:25:24,314] Trial 0 finished with value: 0.8408603407295698 and parameters: {'max_depth': 4, 'min_samples_split': 20, 'min_samples_leaf': 15, 'class_weight': None, 'n_estimators': 198, 'learning_rate': 0.0013927723945289009}. Best is trial 0 with value: 0.8408603407295698.
[I 2025-04-15 20:25:31,194] Trial 1 finished with value: 0.8554033035197184 and parameters: {'max_depth': 9, 'min_samples_split': 13, 'min_samples_leaf': 15, 'class_weight': 'balanced', 'n_estimators': 841, 'learning_rate': 0.0033572967053517922}. Best is trial 1 with value: 0.8554033035197184.
[I 2025-04-15 20:25:32,935] Trial 2 finished with value: 0.8436237087550891 and parameters: {'max_depth': 2, 'min_samples_split': 5, 'min_samples_leaf': 7, 'class_weight': None, 'n_estimators': 326, 'learning_rate': 0.032781876533976156}. Best is trial 1 with value: 0.8554033035197184.
[I 2025-04-15 20:25:34,383] Trial 3 finished w

best_params for main estimator


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "AdaBoostClassifier"


,0
target,no_fragmentation
model,AdaBoostClassifier_no_fragmentation_smote_opt-...
holdout_test_f1_macro,0.891551
holdout_test_accuracy_balanced,0.936364
holdout_test_roc_auc,0.963636
holdout_test_f1,0.851064
holdout_test_accuracy,0.906667
cv_test_f1_macro_median,0.881326
cv_test_accuracy_balanced_median,0.897436
cv_test_roc_auc_median,0.958974


Model saved in ../results/models_modelling4/AdaBoostClassifier_no_fragmentation_smote_opt-model


# Random Forest

In [12]:
estimator = RandomForestClassifier
target = 'splashing'
estimator_params = {}
# params_to_process = ['gamma']
params_to_process = None

def rf_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):

    suggested_estimator_params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "max_depth": trial.suggest_int("max_depth", 2, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int('min_samples_leaf', 1, 20),
        "max_features": trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        "bootstrap": trial.suggest_categorical('bootstrap', [True, False]),
        "criterion": trial.suggest_categorical('criterion', ['gini', 'entropy']),
        "class_weight": trial.suggest_categorical('class_weight', [None, 'balanced'])
    }

    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params,
    )
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=rf_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-15 20:26:18,506] A new study created in memory with name: model_study
[I 2025-04-15 20:26:20,108] Trial 0 finished with value: 0.8354563185744841 and parameters: {'n_estimators': 406, 'max_depth': 20, 'min_samples_split': 15, 'min_samples_leaf': 12, 'max_features': 'sqrt', 'bootstrap': True, 'criterion': 'gini', 'class_weight': None}. Best is trial 0 with value: 0.8354563185744841.
[I 2025-04-15 20:26:21,192] Trial 1 finished with value: 0.863569201181926 and parameters: {'n_estimators': 251, 'max_depth': 5, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': 'sqrt', 'bootstrap': True, 'criterion': 'entropy', 'class_weight': 'balanced'}. Best is trial 1 with value: 0.863569201181926.
[I 2025-04-15 20:26:22,148] Trial 2 finished with value: 0.8702590351921765 and parameters: {'n_estimators': 239, 'max_depth': 11, 'min_samples_split': 13, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'bootstrap': False, 'criterion': 'gini', 'class_weight': 'balanced'}. Best is tria

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "RandomForestClassifier"


,0
target,splashing
model,RandomForestClassifier_splashing_smote_opt-model
holdout_test_f1_macro,0.829027
holdout_test_accuracy_balanced,0.834491
holdout_test_roc_auc,0.89892
holdout_test_f1,0.87234
holdout_test_accuracy,0.84
cv_test_f1_macro_median,0.835913
cv_test_accuracy_balanced_median,0.856037
cv_test_roc_auc_median,0.947619


Model saved in ../results/models_modelling4/RandomForestClassifier_splashing_smote_opt-model


In [13]:
estimator = RandomForestClassifier
target = 'no_fragmentation'
estimator_params = {}
# params_to_process = ['gamma']
params_to_process = None

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=rf_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-15 20:26:42,581] A new study created in memory with name: model_study
[I 2025-04-15 20:26:44,266] Trial 0 finished with value: 0.8246542150819938 and parameters: {'n_estimators': 406, 'max_depth': 20, 'min_samples_split': 15, 'min_samples_leaf': 12, 'max_features': 'sqrt', 'bootstrap': True, 'criterion': 'gini', 'class_weight': None}. Best is trial 0 with value: 0.8246542150819938.
[I 2025-04-15 20:26:45,349] Trial 1 finished with value: 0.8254165514795991 and parameters: {'n_estimators': 251, 'max_depth': 5, 'min_samples_split': 5, 'min_samples_leaf': 7, 'max_features': 'sqrt', 'bootstrap': True, 'criterion': 'entropy', 'class_weight': 'balanced'}. Best is trial 1 with value: 0.8254165514795991.
[I 2025-04-15 20:26:46,407] Trial 2 finished with value: 0.8294019176118625 and parameters: {'n_estimators': 239, 'max_depth': 11, 'min_samples_split': 13, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'bootstrap': False, 'criterion': 'gini', 'class_weight': 'balanced'}. Best is tr

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "RandomForestClassifier"


,0
target,no_fragmentation
model,RandomForestClassifier_no_fragmentation_smote_...
holdout_test_f1_macro,0.888889
holdout_test_accuracy_balanced,0.920455
holdout_test_roc_auc,0.971818
holdout_test_f1,0.844444
holdout_test_accuracy,0.906667
cv_test_f1_macro_median,0.875762
cv_test_accuracy_balanced_median,0.867216
cv_test_roc_auc_median,0.949634


Model saved in ../results/models_modelling4/RandomForestClassifier_no_fragmentation_smote_opt-model


# LightGBM

In [14]:
estimator = LGBMClassifier
target = 'splashing'
estimator_params = {
    'verbose': -1,
    'deterministic':True,
    'force_col_wise':True,
    'njobs': 1,
}
# params_to_process = ['gamma']
params_to_process = None

def lgbm_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):

    suggested_estimator_params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        # "max_depth": trial.suggest_int("max_depth", 1, 10),
        "num_leaves": trial.suggest_int("num_leaves", 4, 512), # more important than max_depth
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "min_split_gain": trial.suggest_float("gamma", 0., 5.), # gamma from XGBoost
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0), # colsample_bytree from XGBoost
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True), # Catboost l2_leaf_reg
        "boosting_type": trial.suggest_categorical("boosting_type", ["gbdt", "dart"]),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
    }

    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params,
    )
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=lgbm_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-15 20:27:12,419] A new study created in memory with name: model_study
[I 2025-04-15 20:27:17,054] Trial 0 finished with value: 0.8590852129479377 and parameters: {'n_estimators': 406, 'learning_rate': 0.22648248189516848, 'num_leaves': 376, 'min_child_samples': 62, 'min_child_weight': 4, 'gamma': 0.7799726016810132, 'subsample': 0.5290418060840998, 'colsample_bytree': 0.9330880728874675, 'reg_alpha': 0.2537815508265665, 'reg_lambda': 0.679657809075816, 'boosting_type': 'dart', 'class_weight': None}. Best is trial 0 with value: 0.8590852129479377.
[I 2025-04-15 20:27:19,332] Trial 1 finished with value: 0.7915149965991307 and parameters: {'n_estimators': 222, 'learning_rate': 0.002846526357761094, 'num_leaves': 158, 'min_child_samples': 55, 'min_child_weight': 9, 'gamma': 1.4561457009902097, 'subsample': 0.8059264473611898, 'colsample_bytree': 0.569746930326021, 'reg_alpha': 0.01474275315991467, 'reg_lambda': 0.029204338471814112, 'boosting_type': 'dart', 'class_weight': 'bal

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LGBMClassifier"


,0
target,splashing
model,LGBMClassifier_splashing_smote_opt-model
holdout_test_f1_macro,0.88
holdout_test_accuracy_balanced,0.868056
holdout_test_roc_auc,0.925154
holdout_test_f1,0.92
holdout_test_accuracy,0.893333
cv_test_f1_macro_median,0.839394
cv_test_accuracy_balanced_median,0.847523
cv_test_roc_auc_median,0.948916


Model saved in ../results/models_modelling4/LGBMClassifier_splashing_smote_opt-model


In [15]:
estimator = LGBMClassifier
target = 'no_fragmentation'
estimator_params = {
    'verbose': -1,
    'deterministic':True,
    'force_col_wise':True,
    'njobs': 1,
}
params_to_process = None

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=lgbm_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-15 20:28:23,730] A new study created in memory with name: model_study
[I 2025-04-15 20:28:28,533] Trial 0 finished with value: 0.8459725380293233 and parameters: {'n_estimators': 406, 'learning_rate': 0.22648248189516848, 'num_leaves': 376, 'min_child_samples': 62, 'min_child_weight': 4, 'gamma': 0.7799726016810132, 'subsample': 0.5290418060840998, 'colsample_bytree': 0.9330880728874675, 'reg_alpha': 0.2537815508265665, 'reg_lambda': 0.679657809075816, 'boosting_type': 'dart', 'class_weight': None}. Best is trial 0 with value: 0.8459725380293233.
[I 2025-04-15 20:28:31,434] Trial 1 finished with value: 0.7957123980199758 and parameters: {'n_estimators': 222, 'learning_rate': 0.002846526357761094, 'num_leaves': 158, 'min_child_samples': 55, 'min_child_weight': 9, 'gamma': 1.4561457009902097, 'subsample': 0.8059264473611898, 'colsample_bytree': 0.569746930326021, 'reg_alpha': 0.01474275315991467, 'reg_lambda': 0.029204338471814112, 'boosting_type': 'dart', 'class_weight': 'bal

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "LGBMClassifier"


,0
target,no_fragmentation
model,LGBMClassifier_no_fragmentation_smote_opt-model
holdout_test_f1_macro,0.857143
holdout_test_accuracy_balanced,0.886364
holdout_test_roc_auc,0.97
holdout_test_f1,0.8
holdout_test_accuracy,0.88
cv_test_f1_macro_median,0.898077
cv_test_accuracy_balanced_median,0.900183
cv_test_roc_auc_median,0.97094


Model saved in ../results/models_modelling4/LGBMClassifier_no_fragmentation_smote_opt-model
